In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
concept_creator_src_path = os.path.join(project_root, 'src', 'concept_creator')
if concept_creator_src_path not in sys.path:
    sys.path.insert(0, concept_creator_src_path)

import time
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Protocol, runtime_checkable
from dotenv import load_dotenv

from concept_creator.src.logic.start_point_picker import StartPointPicker
from concept_creator.src.concept_creation_repository import ConceptCreationRepository
from common.critical_point import CriticalPoint, CriticalPointType
from utils.graph_visualizer import visualize_graph

load_dotenv()

uri = os.getenv("NEO4J_DSN")
user = os.getenv("NEO4J_USER")
password = os.getenv("NEO4J_PASSWORD")
repository = ConceptCreationRepository(uri, user, password)
print(f"Project Root: {project_root}")

In [ ]:
session_id = "3_1"

start_time = time.time()
image_ids = repository.get_image_ids_for_session(session_id)
image_graphs_dict: Dict[str, nx.Graph] = {}
for image_id in image_ids:
    image_graphs_dict[image_id] = repository.get_image_graph(image_id)
image_graphs: List[nx.Graph] = list(image_graphs_dict.values())
elapsed = (time.time() - start_time) * 1000

has_endpoint = all(
    any(nx.degree(g, n) == 1 for n in g.nodes) for g in image_graphs
)
structure_type = "Open" if has_endpoint else "Closed"

print(f"Session: {session_id}")
print(f"Graphs: {len(image_graphs)}")
print(f"Avg nodes: {np.mean([g.number_of_nodes() for g in image_graphs]):.1f}")
print(f"Structure: {structure_type}")
print(f"Loaded in {elapsed:.0f}ms")

In [ ]:
@dataclass
class StartPointResult:
    centroid: np.ndarray
    dominant_label: str
    expected_start_degree: Optional[int]
    structure_type: str
    start_nodes: Dict[int, Any]
    all_clusters: Dict[int, List[CriticalPoint]]
    valid_clusters: Dict[int, List[CriticalPoint]]
    winning_cluster: Optional[List[CriticalPoint]]


@runtime_checkable
class StartPointAlgorithm(Protocol):
    def pick_start_points(self, graphs: List[nx.Graph]) -> StartPointResult: ...


class CurrentStartPointAlgorithm:
    """Wraps existing StartPointPicker with iterative eps-search
    (mirrors CriticalPointConceptService._determine_start_point logic)."""

    def pick_start_points(self, graphs: List[nx.Graph]) -> StartPointResult:
        MAX_ITERATIONS = 15
        start_clustering_eps = 0.01
        eps_step = 0.05
        clustering_algorithm = "optics"

        picker = StartPointPicker(graphs, clustering_algorithm=clustering_algorithm)

        for coeff in np.arange(0.4, 0.8):
            min_samples = int(len(graphs) * coeff)
            eps = start_clustering_eps
            for _ in range(MAX_ITERATIONS):
                if picker.get_start_point_characteristic() is not None:
                    break
                picker.determine_start_point_characteristic(
                    clustering_eps=eps,
                    clustering_min_samples=min_samples,
                )
                eps += eps_step
                min_samples += 1
            if picker.get_start_point_characteristic() is not None:
                break

        characteristic = picker.get_start_point_characteristic()
        if characteristic is None:
            raise ValueError("Could not determine start point characteristic")

        dominant_label, centroid = characteristic
        start_nodes: Dict[int, Any] = {}
        for i, graph in enumerate(graphs):
            start_nodes[i] = picker.get_start_point_for_graph(graph)

        return StartPointResult(
            centroid=centroid,
            dominant_label=dominant_label,
            expected_start_degree=picker.expected_start_degree,
            structure_type=picker.structure_type,
            start_nodes=start_nodes,
            all_clusters=dict(picker.clusters),
            valid_clusters=dict(picker.valid_clusters),
            winning_cluster=picker.final_cluster_points,
        )


def draw_grey_contours(graphs: List[nx.Graph], ax: plt.Axes):
    """Draw all graphs as light grey contours using normalized coordinates.
    For each Vector node, draws a line between its two Point neighbors."""
    for graph in graphs:
        for node_id, data in graph.nodes(data=True):
            labels = data.get("labels", [])
            if "Vector" not in labels:
                continue
            neighbors = list(graph.neighbors(node_id))
            point_coords = []
            for n in neighbors:
                n_data = graph.nodes[n]
                nx_val = n_data.get("normalized_x")
                ny_val = n_data.get("normalized_y")
                if nx_val is not None and ny_val is not None:
                    point_coords.append((nx_val, ny_val))
            if len(point_coords) == 2:
                ax.plot(
                    [point_coords[0][0], point_coords[1][0]],
                    [point_coords[0][1], point_coords[1][1]],
                    color='#CCCCCC', alpha=0.3, linewidth=0.8,
                )


def get_point_nodes(graph: nx.Graph) -> List[dict]:
    """Return Point-type nodes with normalized coords and labels."""
    points = []
    for node_id, data in graph.nodes(data=True):
        labels = data.get("labels", [])
        if "Point" not in labels:
            continue
        nx_val = data.get("normalized_x")
        ny_val = data.get("normalized_y")
        if nx_val is not None and ny_val is not None:
            points.append({"node_id": node_id, "x": nx_val, "y": ny_val, "labels": labels})
    return points


print("Interface + algorithm defined")

In [ ]:
algorithm: StartPointAlgorithm = CurrentStartPointAlgorithm()

start_time = time.time()
result = algorithm.pick_start_points(image_graphs)
elapsed = (time.time() - start_time) * 1000

print(f"Centroid: ({result.centroid[0]:.3f}, {result.centroid[1]:.3f})")
print(f"Dominant label: {result.dominant_label}")
print(f"Expected degree: {result.expected_start_degree}")
print(f"Structure: {result.structure_type}")
print(f"Clusters: {len(result.all_clusters)} total, {len(result.valid_clusters)} valid")
print(f"Start nodes assigned: {len(result.start_nodes)}")
print(f"Elapsed: {elapsed:.0f}ms")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

cluster_ids = sorted(result.all_clusters.keys())
cmap = plt.cm.tab10

for i, cid in enumerate(cluster_ids):
    points = result.all_clusters[cid]
    coords = np.array([p.coordinates for p in points])
    color = cmap(i % 10)
    ax.scatter(coords[:, 0], coords[:, 1], c=[color], s=40, alpha=0.7, label=f"Cluster {cid}")

if result.winning_cluster:
    win_coords = np.array([p.coordinates for p in result.winning_cluster])
    ax.scatter(win_coords[:, 0], win_coords[:, 1], marker='*', s=200,
               c='gold', edgecolors='black', linewidths=0.8, zorder=5, label="Winning")

ax.set_title(f"{session_id} | {result.structure_type} | {len(result.all_clusters)} clusters")
ax.set_xlabel("Normalized X")
ax.set_ylabel("Normalized Y")
ax.invert_yaxis()
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=8)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

draw_grey_contours(image_graphs, ax)

cluster_ids = sorted(result.all_clusters.keys())
cmap = plt.cm.tab10
for i, cid in enumerate(cluster_ids):
    points = result.all_clusters[cid]
    coords = np.array([p.coordinates for p in points])
    color = cmap(i % 10)
    ax.scatter(coords[:, 0], coords[:, 1], c=[color], s=40, alpha=0.7, label=f"Cluster {cid}")

if result.winning_cluster:
    win_coords = np.array([p.coordinates for p in result.winning_cluster])
    ax.scatter(win_coords[:, 0], win_coords[:, 1], marker='*', s=200,
               c='gold', edgecolors='black', linewidths=0.8, zorder=5, label="Winning")

ax.set_title(f"{session_id} | Clusters on graph contours")
ax.set_xlabel("Normalized X")
ax.set_ylabel("Normalized Y")
ax.invert_yaxis()
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=8)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

draw_grey_contours(image_graphs, ax)

# One marker per graph at its selected start node
for graph_idx, node_id in result.start_nodes.items():
    if node_id is None:
        continue
    graph = image_graphs[graph_idx]
    data = graph.nodes[node_id]
    nx_val = data.get("normalized_x")
    ny_val = data.get("normalized_y")
    if nx_val is not None and ny_val is not None:
        ax.scatter(nx_val, ny_val, c='dodgerblue', s=60, alpha=0.7, zorder=4)

# Centroid as black crosshair
ax.scatter(result.centroid[0], result.centroid[1], marker='P', s=200,
           c='black', zorder=5, label="Centroid")

# Centroid direction arrow from origin
centroid_norm = np.linalg.norm(result.centroid)
if centroid_norm > 0:
    ax.annotate("", xy=result.centroid, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.set_title(f"{session_id} | Final start points ({len(result.start_nodes)} graphs)")
ax.set_xlabel("Normalized X")
ax.set_ylabel("Normalized Y")
ax.invert_yaxis()
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=8)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
image_id = image_ids[0]

fig, ax = plt.subplots(figsize=(10, 8))
draw_grey_contours(image_graphs, ax)

# Find graph index for this image_id
graph = image_graphs_dict[image_id]
graph_idx = list(image_graphs_dict.keys()).index(image_id)

# Draw edges of the specific image graph in color
for node_id, data in graph.nodes(data=True):
    labels = data.get("labels", [])
    if "Vector" not in labels:
        continue
    neighbors = list(graph.neighbors(node_id))
    point_coords = []
    for n in neighbors:
        n_data = graph.nodes[n]
        nx_val = n_data.get("normalized_x")
        ny_val = n_data.get("normalized_y")
        if nx_val is not None and ny_val is not None:
            point_coords.append((nx_val, ny_val))
    if len(point_coords) == 2:
        ax.plot(
            [point_coords[0][0], point_coords[1][0]],
            [point_coords[0][1], point_coords[1][1]],
            color='royalblue', alpha=0.8, linewidth=2.0,
        )

# Point nodes colored by type
type_colors = {"EndPoint": "blue", "IntersectionPoint": "red", "CornerPoint": "green"}
for point in get_point_nodes(graph):
    color = "gray"
    for label in point["labels"]:
        if label in type_colors:
            color = type_colors[label]
            break
    ax.scatter(point["x"], point["y"], c=color, s=50, zorder=4, alpha=0.9)

# Highlight selected start point for this image
start_node = result.start_nodes.get(graph_idx)
if start_node is not None:
    sn_data = graph.nodes[start_node]
    sx = sn_data.get("normalized_x")
    sy = sn_data.get("normalized_y")
    if sx is not None and sy is not None:
        ax.scatter(sx, sy, marker='*', s=400, c='gold', edgecolors='black',
                   linewidths=1.5, zorder=6, label="Start point")

# Centroid + direction arrow
ax.scatter(result.centroid[0], result.centroid[1], marker='P', s=200,
           c='black', zorder=5, label="Centroid")
centroid_norm = np.linalg.norm(result.centroid)
if centroid_norm > 0:
    ax.annotate("", xy=result.centroid, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.set_title(f"{session_id} | Image: {image_id}")
ax.set_xlabel("Normalized X")
ax.set_ylabel("Normalized Y")
ax.invert_yaxis()
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=8)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
num_graphs = len(image_graphs)
cols = 3
rows = (num_graphs + cols - 1) // cols

fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axs = axs.flatten() if num_graphs > 1 else [axs]

for i, graph in enumerate(image_graphs):
    start_node = result.start_nodes.get(i)
    visualize_graph(graph, ax=axs[i])

    if start_node is not None:
        node_data = graph.nodes[start_node]
        if 'x' in node_data and 'y' in node_data:
            x_val = node_data['x']
            y_val = node_data['y']
            x = (x_val['min'] + x_val['max']) / 2 if isinstance(x_val, dict) else x_val
            y = (y_val['min'] + y_val['max']) / 2 if isinstance(y_val, dict) else y_val
            axs[i].scatter(x, y, marker='*', s=500, c='red',
                          edgecolors='black', linewidths=1.5, zorder=10)
    axs[i].set_title(f"Graph {i} | Start: {start_node}")

for j in range(num_graphs, rows * cols):
    axs[j].axis('off')

plt.tight_layout()
plt.show()